# SNN Event-Based Gesture Recognition
**Leaky Integrate-and-Fire SNN on DVS Gesture Dataset (IBM, 11 classes)**

This notebook trains a spiking neural network for event-based gesture recognition using the DVS128 Gesture Dataset.
- **Dataset**: IBM DVS Gesture (11 hand gestures, event-camera)
- **Model**: 3-layer LIF network with rate coding
- **Framework**: snnTorch

### Download Dataset
1. Download from Zenodo: https://zenodo.org/records/8060604
2. Place `train.npy` and `test.npy` in `./data/dvs_gesture/`

### Known Limitations
- **FC-only**: ignores spatial structure (ConvSNN would be better)
- **Rate Coding**: wastes temporal precision of event cameras
- **Surrogate Gradient**: Fast Sigmoid approximation introduces mismatch
- **Dead Neurons**: sparse input (~5%) may cause silent LIF neurons

In [ ]:
# Cell 1 — Install dependencies
# Run once, then restart kernel
import subprocess
subprocess.run(['pip', 'install', 'snntorch', '-q'])

In [ ]:
# Cell 2 — Imports
import torch
import torch.nn as nn
import snntorch as snn
from snntorch import surrogate
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device} | snnTorch: {snn.__version__} | PyTorch: {torch.__version__}')

In [ ]:
# Cell 3 — Dataset
# Expects ./data/dvs_gesture/train.npy and test.npy
# Download: https://zenodo.org/records/8060604
#
# If not available, falls back to synthetic data for testing.

T = 20     # timesteps
C, H, W = 2, 32, 32
N_CLASSES = 11

class DVSGestureDataset(Dataset):
    """DVS Gesture Dataset loader (Zenodo pre-processed .npy format)."""
    def __init__(self, path, T=20):
        data = np.load(path, allow_pickle=True).item()
        self.X = torch.tensor(data['frames'], dtype=torch.float32)  # [N, T, C, H, W]
        self.y = torch.tensor(data['labels'], dtype=torch.long)     # [N]
        self.T = T
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i, :self.T], self.y[i]

class SyntheticDataset(Dataset):
    """Synthetic DVS-style data for pipeline testing (no real dataset needed)."""
    def __init__(self, n=1000, T=20, C=2, H=32, W=32, n_classes=11):
        self.X = torch.bernoulli(torch.rand(n, T, C, H, W) * 0.05)
        self.y = torch.randint(0, n_classes, (n,))
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

DATA_PATH = Path('./data/dvs_gesture')

if (DATA_PATH / 'train.npy').exists():
    print('Loading real DVS Gesture dataset...')
    train_set = DVSGestureDataset(DATA_PATH / 'train.npy', T=T)
    test_set  = DVSGestureDataset(DATA_PATH / 'test.npy',  T=T)
    REAL_DATA = True
else:
    print('DVS dataset not found — using synthetic fallback.')
    print('Download: https://zenodo.org/records/8060604')
    train_set = SyntheticDataset(n=1000)
    test_set  = SyntheticDataset(n=200)
    REAL_DATA = False

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=32, shuffle=False)
print(f'Train: {len(train_set)} | Test: {len(test_set)} | Real data: {REAL_DATA}')

In [ ]:
# Cell 4 — Model
spike_grad = surrogate.fast_sigmoid(slope=25)

class EventSNN(nn.Module):
    """
    3-layer LIF SNN for event-based gesture classification.
    Architecture: FC(2048->512) -> LIF -> FC(512->128) -> LIF -> FC(128->11) -> LIF
    Temporal integration: rate coding (spike sum over T timesteps)

    Limitations:
    - FC-only: ignores spatial structure of DVS frames
    - Rate coding: does not exploit precise spike timing
    - Surrogate gradient: approximation of true Heaviside derivative
    """
    def __init__(self, input_dim=C*H*W, h1=512, h2=128, n_classes=N_CLASSES, beta=0.9):
        super().__init__()
        self.T = T
        self.fc1  = nn.Linear(input_dim, h1)
        self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad)
        self.fc2  = nn.Linear(h1, h2)
        self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad)
        self.fc3  = nn.Linear(h2, n_classes)
        self.lif3 = snn.Leaky(beta=beta, spike_grad=spike_grad)

    def forward(self, x):  # x: [B, T, C, H, W]
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()
        spk_out, mem_out = [], []
        for t in range(self.T):
            inp        = x[:, t].float().flatten(1)
            spk1, mem1 = self.lif1(self.fc1(inp),  mem1)
            spk2, mem2 = self.lif2(self.fc2(spk1), mem2)
            spk3, mem3 = self.lif3(self.fc3(spk2), mem3)
            spk_out.append(spk3)
            mem_out.append(mem3)
        return torch.stack(spk_out).sum(0), torch.stack(mem_out)

net = EventSNN().to(device)
print(net)
print(f'\nParameters: {sum(p.numel() for p in net.parameters()):,}')

In [ ]:
# Cell 5 — Training
optimizer = torch.optim.Adam(net.parameters(), lr=2e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
loss_fn   = nn.CrossEntropyLoss()

N_EPOCHS = 10
history  = {'train_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(N_EPOCHS):
    net.train()
    epoch_loss, epoch_correct, epoch_total = 0, 0, 0
    for data, targets in train_loader:
        data, targets = data.to(device), targets.to(device)
        spk_out, _ = net(data)
        loss = loss_fn(spk_out, targets)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        epoch_loss    += loss.item()
        epoch_correct += spk_out.argmax(1).eq(targets).sum().item()
        epoch_total   += targets.size(0)
    scheduler.step()

    # Validation
    net.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for data, targets in test_loader:
            data, targets = data.to(device), targets.to(device)
            spk_out, _ = net(data)
            val_correct += spk_out.argmax(1).eq(targets).sum().item()
            val_total   += targets.size(0)

    train_acc = epoch_correct / epoch_total * 100
    val_acc   = val_correct   / val_total   * 100
    avg_loss  = epoch_loss / len(train_loader)
    history['train_loss'].append(avg_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    print(f'Epoch {epoch+1:2d}/{N_EPOCHS} | Loss: {avg_loss:.4f} | Train: {train_acc:.1f}% | Val: {val_acc:.1f}%')

print('\nTraining complete!')

In [ ]:
# Cell 6 — Plots
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('EventSNN — DVS Gesture Recognition', fontsize=13, fontweight='bold')

axes[0].plot(history['train_loss'], 'o-', color='#7c6af7', lw=2.5, ms=7)
axes[0].set(xlabel='Epoch', ylabel='Cross-Entropy Loss', title='Training Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_acc'], 'o-', color='#7c6af7', lw=2.5, ms=7, label='Train')
axes[1].plot(history['val_acc'],   's-', color='#e07b39', lw=2.5, ms=7, label='Val')
axes[1].axhline(100/N_CLASSES, color='red', ls='--', label=f'Chance ({100/N_CLASSES:.1f}%)')
axes[1].set(xlabel='Epoch', ylabel='Accuracy (%)', title='Train vs Val Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Membrane potential visualisation
net.eval()
sample, _ = next(iter(test_loader))
with torch.no_grad():
    _, mem_rec = net(sample[:1].to(device))
mem_np = mem_rec[:, 0, :].cpu().numpy()
im = axes[2].imshow(mem_np.T, aspect='auto', cmap='plasma', origin='lower')
fig.colorbar(im, ax=axes[2], label='Membrane V')
axes[2].set(xlabel='Timestep', ylabel='Output Neuron', title='LIF Membrane Potential')

plt.tight_layout()
plt.savefig('snn_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: snn_results.png')

In [ ]:
# Cell 7 — Save checkpoint
import json, os
os.makedirs('checkpoints', exist_ok=True)
torch.save({'model_state': net.state_dict(),
            'history': history,
            'config': {'T': T, 'C': C, 'H': H, 'W': W,
                       'n_classes': N_CLASSES, 'real_data': REAL_DATA}},
           'checkpoints/eventsnn_ckpt.pt')
with open('checkpoints/results.json', 'w') as f:
    json.dump({'final_val_acc': history['val_acc'][-1],
               'best_val_acc':  max(history['val_acc']),
               'real_data':     REAL_DATA}, f, indent=2)
print('Checkpoint saved!')
print(f'Best Val Acc: {max(history["val_acc"]):.1f}%')